# 00 — BioLLM Post-Training: Research Overview

## Objective

Define the research questions, explain the biomedical data and their origin, trace the complete processing and training pipeline, and verify the boundary between experimentation and deployment. This technical roadmap distinguishes completed local runs, illustrative demonstrations, and unverified deployment paths.

## Why this study matters

A **Large Language Model (LLM)** produces token sequences, but a biomedical research assistant must also select evidence-supported answers and avoid unsupported claims. **Post-training** changes behavior after pretraining rather than learning language from scratch. Three families of training signals are compared:

- **SFT (Supervised Fine-Tuning):** learn to generate a reference response given a question and context.
- **DPO (Direct Preference Optimization):** learn to prefer a chosen completion over a rejected one without separately fitting a reward model for that optimization step.
- **GRPO (Group Relative Policy Optimization):** sample several completions and use their relative rewards to update the model.

**QLoRA (Quantized Low-Rank Adaptation)** provides a memory-conscious means of training adapters without updating all of the pretrained parameters. A separate reward-model example trains preference scores, while a simplified **PPO (Proximal Policy Optimization)** cell in notebook 10 illustrates a clipped policy update. That numerical example is *not* full **RLHF (Reinforcement Learning from Human Feedback)**.

## Hypotheses and model comparisons

| ID | Hypothesis to test | Required comparison |
|---|---|---|
| H1 | Supervision can improve answer quality | Starting checkpoint vs. SFT |
| H2 | Preference optimization can change preferred-response behavior | Shared SFT checkpoint vs. SFT + DPO |
| H3 | Verifiable-reward training can improve selected task metrics | Shared SFT checkpoint vs. SFT + GRPO |
| H4 | Incomplete reward specifications permit shortcut behavior | Weak formatting reward vs. stricter answer check |

These are falsifiable questions. A later checkpoint is not considered better until its outputs are measured on held-out records.

## Original data sources and retrieval

The repository uses real public datasets where model-training results are intended; it also includes explicitly synthetic **concept demonstrations**. Downloaded inputs remain in the local research workspace, not the deployment package.

### 1. PubMedQA: biomedical question answering

- **Original dataset:** [qiaojin/PubMedQA on Hugging Face](https://huggingface.co/datasets/qiaojin/PubMedQA). The [original research repository](https://github.com/pubmedqa/pubmedqa) contains the benchmark resources.
- **Used configuration:** `pqa_labeled`, containing approximately 1,000 expert-labeled yes/no/maybe examples in a source `train` split. There are separate `pqa_artificial` and `pqa_unlabeled` configurations; do not treat unlabeled records as ground truth.
- **Important fields:** `pubid`, `question`, `context` (a nested dictionary with `contexts`, `labels`, and related fields), `long_answer`, and `final_decision`.
- **Role in pipeline:** supervised training, starting-model validation, GRPO answer rewards, and cross-checking outputs.
- **Split policy:** notebook 03 produces a stable approximately 80/10/10 split by hashing each publication ID with SHA-256 (Secure Hash Algorithm 256-bit) and stores exact IDs plus a source fingerprint in `data/pubmedqa_split_manifest.json`. All model notebooks verify this manifest. The dataset's official evaluation protocol uses a separate 500-item test selection. Our exploratory estimates are **not official leaderboard scores**. The saved manifest verifies exact publication-ID separation; semantic near-duplicates still require review before strong scientific claims.

### 2. Med-REFL-DPO: automatically constructed preferences

- **Original dataset:** [HANI-LAB/Med-REFL-DPO on Hugging Face](https://huggingface.co/datasets/HANI-LAB/Med-REFL-DPO); [research code](https://github.com/TianYin123/Med-REFL).
- **Available configurations:** `reasoning_enhancement` and `reflection_enhancement`.
- **Role in pipeline:** training a Direct Preference Optimization policy and a pairwise scalar reward model.
- **Caution:** preferred/rejected trajectories were constructed automatically by the dataset creators using reasoning-generation and evaluation methods. They are not equivalent to human preference annotations. Inspect actual data fields before mapping records into chosen/rejected completions.

### 3. TCGA-LUAD: optional genomics evidence

- **Original program:** [The Cancer Genome Atlas Lung Adenocarcinoma (TCGA-LUAD) at the Genomic Data Commons](https://portal.gdc.cancer.gov/projects/TCGA-LUAD).
- **Prepared public tables:** [University of California, Santa Cruz Xena data portal](https://xena.ucsc.edu/download-data/) provides cohort expression, clinical, and survival tables; verify the exact files and their revisions prior to analysis.
- **Role here:** notebook 14 currently tests a schema and rules for interpreting evidence from a scientific workflow. Its numeric example is explicitly illustrative. A full TCGA analysis has **not** been implemented by that demonstration.

### Getting the original data

The following cell runs only when `RUN_PUBLIC_DATA=1`. It fetches the two public Hugging Face datasets and prints their actual field names and one abstract excerpt. Successful loading caches source files locally. No example records are committed to Git.

In [1]:
# Purpose: Optionally download original PubMedQA and Med-REFL-DPO records and inspect their
#   schemas.
# Result: Dataset sizes, column names, and a short original evidence excerpt; otherwise an
#   explicit skip message.
import os

if os.getenv("RUN_PUBLIC_DATA", "0") == "1":
    from datasets import load_dataset

    pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    preferences = load_dataset("HANI-LAB/Med-REFL-DPO", "reasoning_enhancement", split="train")
    print("PubMedQA records:", len(pubmedqa), "fields:", pubmedqa.column_names)
    print("PubMedQA evidence excerpt:", " ".join(pubmedqa[0]["context"]["contexts"])[:350])
    print("Med-REFL-DPO records:", len(preferences), "fields:", preferences.column_names)
    # Keep downloads in the Hugging Face cache; see README.md for optional save_to_disk() calls.
else:
    print("Public data unchanged. To fetch originals, set RUN_PUBLIC_DATA=1 before running this cell.")

Public data unchanged. To fetch originals, set RUN_PUBLIC_DATA=1 before running this cell.


### Locate the project workspaces

**Procedure.** Resolve the current directory, locate `exploration/`, and create its ignored `runs/` directory.

**Reading the result.** The printed directory is the location where experimental artifacts will be written; no model training occurs.

In [2]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


## Whole-project architecture

```text
  ORIGINAL PUBLIC DATA
   PubMedQA (questions + article abstract + labeled decision)
   Med-REFL-DPO (preferred/rejected generated medical reasoning)
   TCGA-LUAD / Xena (optional cohort evidence, not yet a complete analysis)
                      |
                      v
  03 / 07  DATA SCHEMA AND SPLITS
           inspect source identifiers, fields, label frequencies,
           preference pairs, duplicate groups, and dataset revisions
                      |
                      v
  04       STARTING-MODEL BASELINE
           Qwen2.5-Instruct -> native chat-template prompt -> deterministic raw output -> strict label -> diagnostics
                      |
                      v
  05 / 06  SUPERVISED FINE-TUNING
           PubMedQA labeled data -> unquantized LoRA adapter (four-bit QLoRA optional, unexecuted) -> held-out evaluation
                      |
                +-----+-------+
                |             |
                v             v
  08 DPO: preference pairs     11 GRPO: sampled completions,
  trained from SFT             programmatic yes/no/maybe reward,
  (Med-REFL-DPO)               trained from SFT (PubMedQA)
                |             |
                +------+------+
                       |
                       v
  12 / 17 / 18  COMPARISONS AND ROBUSTNESS
                saved row-level responses -> metric tables -> paired tests
                -> reward-hacking checks -> reproducible summary
                       |
                       v
  13 / 15       EXPORT AND RELEASE CHECKS
                verify checkpoint, model identity, evaluation and artifacts
                       |
                       v
  deployment/   INDEPENDENT INFERENCE RUNTIME
                integrated app.py + lightweight dependencies + Dockerfile
```

### The training variants are related but not identical experiments

The untouched instruction-tuned Qwen model is the starting-model baseline. SFT produces a shared adapter, and **both executed DPO and GRPO runs initialize from that SFT result**. DPO learns from automatically constructed preference data; GRPO uses correctness/format rewards against labeled biomedical decisions. Therefore training datasets and computational costs differ even though the starting checkpoint is controlled. Evaluate claims about individual methods only with suitable controls and report any domain mismatch.

### What each phase produces

| Notebooks | Computation | Expected output and interpretation |
|---|---|---|
| 00–03 | Workspace checks, memory estimates, mathematical examples, public schemas | Paths, sample gradients, data-field lists; most are descriptive or illustrative |
| 04 | Starting checkpoint inference | Completed 121-record validation predictions; 19 correct |
| 05–06 | SFT and adapter evaluation | Completed unquantized LoRA adapter and 73/121 validation result |
| 07–09 | Public preference inspection, DPO, reward modeling | DPO adapter completed; reward model failed independent preference generalization |
| 10–11 | PPO-style bandit demonstration, GRPO reward-based updates | Synthetic PPO demonstration and completed GRPO adapter |
| 12 | Comparative validation | Matched 121-row base/SFT/DPO/GRPO comparison and paired uncertainty |
| 13–15 | Export, evidence contract, MLOps gates | Merged SFT Hugging Face checkpoint; illustrative interface; research registry with no promoted model |
| 16–17 | Technical Q&A and reward robustness | Completed evidence synthesis and bounded validation perturbations |

## Experimental controls and limitations

A convincing comparison requires the same evaluation prompts, model base version, parsing and decoding conventions, stable example IDs, and recorded random seeds. Source revisions and training configurations must be captured. Group related records by paper/case where needed, to avoid training and validation leakage. Public medical preference pairs and PubMedQA answer labels are different supervision signals. A higher training reward, lower loss, or a new adapter file does not establish better scientific correctness.

The examples in this repository distinguish **illustrative demonstrations**, **implemented optional procedures**, and **measured experiments**. Model quality claims require actual saved predictions and independently calculated metrics. No claims of clinical use, diagnosis, causal proof, or completed TCGA-based biomarker validation follow from these notebooks.

## Project organization and execution

```text
BioLLM-PostTraining-Lab/
├── README.md                         # Detailed instructions and dataset sources
├── exploration/
│   ├── notebooks/                    # Research code, explanations and comparisons
│   ├── data/                         # Local downloaded inputs (not committed)
│   ├── runs/                         # Saved adapters and predictions (not committed)
│   ├── results/                      # Locally generated tables/reports
│   └── requirements.txt
└── deployment/
    ├── app.py                        # Standalone service implementation
    ├── Dockerfile
    ├── requirements.txt
    └── model/                        # Explicitly approved model artifact location
```

### How to execute

Install packages using `exploration/requirements.txt`, then open notebooks `00`–`17` in numerical order; notebook 18 is a separate local-only report step. The default small model is `Qwen/Qwen2.5-0.5B-Instruct`; larger models require deliberate resource evaluation. The following environment switches are recognized by different notebooks:

| Control | Purpose |
|---|---|
| `RUN_PUBLIC_DATA=1` | Fetch public dataset samples (00, 03, 07) |
| `RUN_TRAINING=1` | Train model adapters (05, 08, 09, 11) |
| `RUN_MODEL_EVAL=1` | Generate validation predictions (04, 06, 12) |
| `RUN_TEST_EVAL=1` and `EXPERIMENT_FROZEN=1` | Allow final held-out evaluation after model selection (18) |
| `RUN_EXPORT=1` | Merge a completed SFT adapter for export (13) |
| `RUN_PACKAGE=1` | Copy permitted saved model files for a deployment handoff (15) |

Do not enable expensive execution modes without confirming GPU memory, source revisions, model compatibility, and test-set protections. Instructions and download examples are expanded in the root [README](../../README.md).

### Inspect the two workspaces

**Procedure.** Check that research notebooks and output folders exist, and whether the separate deployment directory is available.

**Reading the result.** `True` means a directory exists; it does not indicate that models have been trained or deployed.

In [3]:
# Purpose: Check that research notebooks and output folders exist, and whether the separate
#   deployment directory is available.
# Result: `True` means a directory exists; it does not indicate that models have been
#   trained or deployed.
from pathlib import Path
for folder in ['notebooks', 'runs', 'results']:
    print(f'{folder:10}', (RESEARCH / folder).exists())
print('Deployment is intentionally separate:', (RESEARCH.parent/'deployment').exists())

notebooks  True
runs       True
results    True
Deployment is intentionally separate: True


### Results

The workspace checks found the notebook, run, result, and separate deployment directories. Public-data retrieval was disabled in this overview execution.

### Interpretation

These cells verify repository organization only. Later notebooks and saved manifests establish the data and model results.

### Limitations

Directory existence does not verify a deployment, dataset revision, checkpoint quality, or clinical suitability.

### Key Observations

The completed SFT route uses unquantized LoRA; the final PubMedQA test remains sealed. See the verified-results summary below for the research outcomes.


## Verified research state

**Results.** On the 121-question PubMedQA validation set, base/SFT/DPO/GRPO answered 19/73/77/76 correctly; the majority-yes rule answered 64 correctly. Every adapted model missed all 17 true-maybe records. Notebook 09's revised reward checkpoints each scored 9/22 on a separate group-disjoint preference test, versus 11/22 for the starting reward adapter. Notebook 10's PPO result is synthetic; notebook 13 verified a merged Hugging Face SFT checkpoint; notebook 15's registry promotes no model.

**Interpretation.** SFT improved on a weak base. DPO's four-case and GRPO's three-case net increments over SFT do not establish a robust winner. The reward model is not a validated RLHF signal.

**Limitations.** Validation has been inspected repeatedly, the PubMedQA final 104 records remain sealed, preference rankings are automatically constructed, and no live deployment or clinical result has been verified.

**Key Observations.** Keep training, validation, preference-test, synthetic, and deployment evidence separate when interpreting this project.

## Applied systems: retrieval, agents, and deployment decisions

Post-training quality is only one part of a dependable biomedical assistant. A retrieval and agent runtime must find suitable evidence, choose permitted tools, recover from interrupted workflows, and measure actual task success after deployment. The following additional topics connect the model experiments to those system-level concerns:

| Engineering question | Where it is addressed | Demonstration status |
|---|---|---|
| Retrieve relevant evidence and measure ranking quality | [04 — Baseline Measurement](04_baseline_evaluation.ipynb) | Illustrative metric-unit tests and optional source-backed TF–IDF matching proxy |
| Avoid unsupported answers and diagnose irrelevant retrieval | [14 — Evidence-Grounded Application](14_oncolncai_integration_demo.ipynb) | Simple evidence selection and abstention checks; no deployed vector index |
| Decide which tool to call and apply execution limits | [14 — Evidence-Grounded Application](14_oncolncai_integration_demo.ipynb) | Deterministic allowlisted tool routing; no live MCP server |
| Understand LangChain and LangGraph reliability tradeoffs | [14](14_oncolncai_integration_demo.ipynb) and [15](15_mlops_pipeline.ipynb) | Architecture discussion and a research registry with in-memory rollback demonstration |
| Manage serving cost, latency, security, and monitoring | [15 — Reproducibility and Packaging](15_mlops_pipeline.ipynb) | Local checkpoint smoke test, closed release gate, and monitoring design |
| Examine prompt injection and reward hacking | [17 — Reward Hacking and Robustness](17_reward_hacking_and_robustness.ipynb) | Saved four-model proxy analysis, six-question model probe, and deterministic boundary checks |
| Read complete technical Q&A | [16 — Technical Decisions](16_technical_discussion.ipynb) | 25 technical Q&A entries linked to actual runs and system demonstrations |

The additional exercises use invented records only where necessary to teach a specific mechanism. Any performance claim requires a real held-out benchmark and saved experiment records.

**Evaluation note:** Notebook 04 has a separate TF–IDF abstract-matching proxy, enabled with `RUN_RETRIEVAL_EVAL=1`. Toy ranking checks are not a production retrieval benchmark; they establish only that the formulas work. Real model validation now uses the instruction-tuned chat template and a frozen publication-ID manifest.


## Key observations and technical discussion
A useful research claim reports baseline, experimental controls, uncertainty, and limitations. The deployment runtime must not be imported by the notebooks.

## Q&A
**Question:** Why retain an untouched starting-model baseline?

**Answer:** It separates the effect of any post-training stage from behavior the instruction-tuned model already had.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.